In [1]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import SAGEConv
from torch_geometric.transforms import NormalizeFeatures

In [2]:
dataset = Planetoid(root='data/Cora', name='Cora', transform=NormalizeFeatures())
data = dataset[0] 

Processing...
C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch_geometric\io\planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle.load(f, encoding='latin1')
C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch_geometric\io\planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle.load(f, encoding='latin1')
C:\Users\ROG\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torch_geometric\io\planetoid.py:107: VisibleDepre

In [3]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # Layer 1: Aggregates features from neighbors
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        # Layer 2: Maps to the number of classes
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GraphSAGE(dataset.num_features, 16, dataset.num_classes).to(device)
data = data.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [5]:
def train():
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    # Only use training nodes for loss calculation
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


In [7]:
@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(dim=1)
    # Calculate accuracy for the test set mask
    correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
    acc = int(correct) / int(data.test_mask.sum())
    return acc

for epoch in range(1, 201):
    loss = train()
    if epoch % 20 == 0:
        test_acc = test()
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Test Acc: {test_acc:.4f}')


Epoch: 020, Loss: 0.1080, Test Acc: 0.8050
Epoch: 040, Loss: 0.1425, Test Acc: 0.8090
Epoch: 060, Loss: 0.1057, Test Acc: 0.8010
Epoch: 080, Loss: 0.0986, Test Acc: 0.8080
Epoch: 100, Loss: 0.1071, Test Acc: 0.8110
Epoch: 120, Loss: 0.1158, Test Acc: 0.8070
Epoch: 140, Loss: 0.1185, Test Acc: 0.8030
Epoch: 160, Loss: 0.0816, Test Acc: 0.8070
Epoch: 180, Loss: 0.1051, Test Acc: 0.8070
Epoch: 200, Loss: 0.0905, Test Acc: 0.8010
